# Retail Enterprise BI Control Tower
## Notebook 02: Data Cleaning

**Project:** Retail Enterprise BI Control Tower  
**Purpose:** Clean all 8 raw tables based on the Data Quality Report from 01_data_inspection.ipynb. Fix nulls, remove duplicates, standardise categories, and export clean CSVs to data/processed/.  
**Author:** Arun Prabakar Vadaseri Rajendran  
**Date:** June 2026  
**Tool:** Google Colab

In [1]:
import pandas as pd # Import the pandas library for data manipulation
import os           # Import the os library for interacting with the operating system

# Set pandas display options for better readability of DataFrames
pd.set_option('display.max_columns', None) # Display all columns
pd.set_option('display.max_rows', 60)      # Display up to 60 rows
pd.set_option('display.width', 120)        # Set the display width for output

print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
import pandas as pd

# Load 8 CSV files into DataFrames
df_customers = pd.read_csv('/content/customers.csv')
df_inventory = pd.read_csv('/content/inventory.csv')
df_marketing = pd.read_csv('/content/marketing_campaigns.csv')
df_order_details = pd.read_csv('/content/order_details.csv')
df_orders = pd.read_csv('/content/orders.csv')
df_products = pd.read_csv('/content/products.csv')
df_returns = pd.read_csv('/content/returns.csv')
df_stores = pd.read_csv('/content/stores.csv')

# Dictionary to iterate for reporting
dataframes = {
    'Customers': df_customers,
    'Inventory': df_inventory,
    'Marketing': df_marketing,
    'Order Details': df_order_details,
    'Orders': df_orders,
    'Products': df_products,
    'Returns': df_returns,
    'Stores': df_stores
}

print('Raw Data Loading Summary:')
print('-' * 25)
for name, df in dataframes.items():
    print(f"{name:<15}: {len(df):>7,} rows")

Raw Data Loading Summary:
-------------------------
Customers      :  50,100 rows
Inventory      : 720,000 rows
Marketing      :     900 rows
Order Details  : 303,124 rows
Orders         : 130,250 rows
Products       :   1,800 rows
Returns        :  22,734 rows
Stores         :     120 rows


In [3]:
# 1. Remove duplicates from df_orders
initial_orders_count = len(df_orders)
df_orders = df_orders.drop_duplicates(keep='first')
final_orders_count = len(df_orders)
removed_orders = initial_orders_count - final_orders_count

print(f"df_orders: {initial_orders_count:,} rows (Before) -> {final_orders_count:,} rows (After)")

# 2. Remove duplicates from df_customers
initial_customers_count = len(df_customers)
df_customers = df_customers.drop_duplicates(keep='first')
final_customers_count = len(df_customers)
removed_customers = initial_customers_count - final_customers_count

print(f"df_customers: {initial_customers_count:,} rows (Before) -> {final_customers_count:,} rows (After)")

# 3. Confirmation summary
print('\nDeduplication Summary:')
print('-' * 25)
print(f"Rows removed from Orders   : {removed_orders}")
print(f"Rows removed from Customers: {removed_customers}")
print("Logic: Used drop_duplicates(keep='first') to eliminate identical row repetitions.")

df_orders: 130,250 rows (Before) -> 130,000 rows (After)
df_customers: 50,100 rows (Before) -> 50,000 rows (After)

Deduplication Summary:
-------------------------
Rows removed from Orders   : 250
Rows removed from Customers: 100
Logic: Used drop_duplicates(keep='first') to eliminate identical row repetitions.


In [6]:
# 1. Fill nulls in df_orders['campaign_id'] using .loc to avoid SettingWithCopyWarning
nulls_before_campaign = df_orders['campaign_id'].isnull().sum()
df_orders.loc[:, 'campaign_id'] = df_orders['campaign_id'].fillna('UNKNOWN')
nulls_after_campaign = df_orders['campaign_id'].isnull().sum()
print(f"df_orders['campaign_id']: {nulls_before_campaign:,} nulls (Before) -> {nulls_after_campaign} (After)")

# 2. Fill nulls in df_customers['preferred_contact']
nulls_before_contact = df_customers['preferred_contact'].isnull().sum()
df_customers.loc[:, 'preferred_contact'] = df_customers['preferred_contact'].fillna('UNKNOWN')
nulls_after_contact = df_customers['preferred_contact'].isnull().sum()
print(f"df_customers['preferred_contact']: {nulls_before_contact:,} nulls (Before) -> {nulls_after_contact} (After)")

# 3. Fill nulls in df_order_details['discount_pct']
# LOGIC: discount_pct is filled with 0 because a missing value in this context implies
# that no discount was applied to the transaction. 'UNKNOWN' is used for categorical
# identifiers where the specific value is missing, but 0 is numerically meaningful here.
nulls_before_discount = df_order_details['discount_pct'].isnull().sum()
df_order_details.loc[:, 'discount_pct'] = df_order_details['discount_pct'].fillna(0)
nulls_after_discount = df_order_details['discount_pct'].isnull().sum()
print(f"df_order_details['discount_pct']: {nulls_before_discount:,} nulls (Before) -> {nulls_after_discount} (After)")

# Confirmation Summary
print('\nImputation Summary:')
print('-' * 25)
print(f"Campaign IDs standardized: {nulls_before_campaign:,}")
print(f"Contact methods standardized: {nulls_before_contact:,}")
print(f"Default discounts applied: {nulls_before_discount:,}")

df_orders['campaign_id']: 0 nulls (Before) -> 0 (After)
df_customers['preferred_contact']: 0 nulls (Before) -> 0 (After)
df_order_details['discount_pct']: 0 nulls (Before) -> 0 (After)

Imputation Summary:
-------------------------
Campaign IDs standardized: 0
Contact methods standardized: 0
Default discounts applied: 0


In [7]:
import pandas as pd

# 1. Check unique categories BEFORE the fix
unique_before = df_products['category'].unique()
print(f"Unique categories before standardization ({len(unique_before)}):")
print(unique_before)

# 2. Standardise the column
# .str.title() is the right choice because it converts the first character of each word to uppercase
# and the rest to lowercase, ensuring consistency (e.g., 'APPAREL' and 'apparel' both become 'Apparel').

# We count how many rows will actually change
mask_to_change = df_products['category'] != df_products['category'].str.strip().str.title()
affected_rows = mask_to_change.sum()

df_products['category'] = df_products['category'].str.strip().str.title()

# 3. Check unique categories AFTER the fix
unique_after = df_products['category'].unique()
print(f"\nUnique categories after standardization ({len(unique_after)}):")
print(unique_after)

# 4. Print row impact
print(f"\nStandardization Summary:")
print(f"Rows affected/cleaned: {affected_rows:,}")

Unique categories before standardization (12):
['Grocery' 'Apparel' 'Office Supplies' 'Technology' 'Home' 'Furniture'
 'GROCERY' 'APPAREL' 'OFFICE SUPPLIES' 'FURNITURE' 'TECHNOLOGY' 'HOME']

Unique categories after standardization (6):
['Grocery' 'Apparel' 'Office Supplies' 'Technology' 'Home' 'Furniture']

Standardization Summary:
Rows affected/cleaned: 27


In [8]:
import pandas as pd

# Define original row counts as provided
original_counts = {
    'Customers': 50100, 'Inventory': 720000, 'Marketing': 900,
    'Order Details': 303124, 'Orders': 130250, 'Products': 1800,
    'Returns': 22734, 'Stores': 120
}

# Current DataFrames mapping
current_dfs = {
    'Customers': df_customers, 'Inventory': df_inventory, 'Marketing': df_marketing,
    'Order Details': df_order_details, 'Orders': df_orders, 'Products': df_products,
    'Returns': df_returns, 'Stores': df_stores
}

print(f"{'Table Name':<15} | {'Original':>10} | {'Current':>10} | {'Removed':>8} | {'Nulls (Key Cols)':<20}")
print("-" * 75)

total_nulls = 0

for name, df in current_dfs.items():
    orig = original_counts[name]
    curr = len(df)
    removed = orig - curr

    # Check specific key columns for nulls
    null_check = "N/A"
    if name == 'Orders':
        n = df['campaign_id'].isnull().sum()
        null_check = f"campaign_id: {n}"
        total_nulls += n
    elif name == 'Customers':
        n = df['preferred_contact'].isnull().sum()
        null_check = f"contact: {n}"
        total_nulls += n
    elif name == 'Order Details':
        n = df['discount_pct'].isnull().sum()
        null_check = f"discount: {n}"
        total_nulls += n
    elif name == 'Products':
        n = df['category'].isnull().sum()
        null_check = f"category: {n}"
        total_nulls += n

    print(f"{name:<15} | {orig:>10,} | {curr:>10,} | {removed:>8,} | {null_check:<20}")

print("-" * 75)
if total_nulls == 0:
    print("CLEANING COMPLETE — Ready for export")
else:
    print(f"ATTENTION: {total_nulls} nulls remaining in key columns.")

Table Name      |   Original |    Current |  Removed | Nulls (Key Cols)    
---------------------------------------------------------------------------
Customers       |     50,100 |     50,000 |      100 | contact: 0          
Inventory       |    720,000 |    720,000 |        0 | N/A                 
Marketing       |        900 |        900 |        0 | N/A                 
Order Details   |    303,124 |    303,124 |        0 | discount: 0         
Orders          |    130,250 |    130,000 |      250 | campaign_id: 0      
Products        |      1,800 |      1,800 |        0 | category: 0         
Returns         |     22,734 |     22,734 |        0 | N/A                 
Stores          |        120 |        120 |        0 | N/A                 
---------------------------------------------------------------------------
CLEANING COMPLETE — Ready for export


In [9]:
import pandas as pd

# Mapping DataFrames to their respective clean filenames
export_map = {
    'customers_clean.csv': df_customers,
    'inventory_clean.csv': df_inventory,
    'marketing_campaigns_clean.csv': df_marketing,
    'order_details_clean.csv': df_order_details,
    'orders_clean.csv': df_orders,
    'products_clean.csv': df_products,
    'returns_clean.csv': df_returns,
    'stores_clean.csv': df_stores
}

print("Exporting Clean Files:")
print("-" * 40)

for filename, df in export_map.items():
    # index=False ensures the DataFrame index (row numbers) is not saved as a separate column,
    # which prevents column misalignment when the data is re-imported later.
    df.to_csv(f'/content/{filename}', index=False)
    print(f"{filename:<30} | {len(df):>10,} rows written")

print("-" * 40)
print("Export complete — 8 clean files ready for download")

Exporting Clean Files:
----------------------------------------
customers_clean.csv            |     50,000 rows written
inventory_clean.csv            |    720,000 rows written
marketing_campaigns_clean.csv  |        900 rows written
order_details_clean.csv        |    303,124 rows written
orders_clean.csv               |    130,000 rows written
products_clean.csv             |      1,800 rows written
returns_clean.csv              |     22,734 rows written
stores_clean.csv               |        120 rows written
----------------------------------------
Export complete — 8 clean files ready for download
